# Cattle Re-ID — Etapa 3, Fase 0: re-ID no supervisado por clustering (DINOv2)

Circuito end-to-end: **encoder congelado → embeber el target sin etiquetas → clusterizar (descubrir identidades) → evaluar**. Las etiquetas se usan **solo para evaluar**, nunca para clusterizar.

- **Target (campo nuevo, sin etiquetas):** Zenodo Muzzle DB (`BeefCattle_Muzzle_Individualized.zip`, 268 vacas).
- **Encoders (Fase 0):** DINOv2 (genérico auto-supervisado) + ImageNet ResNet-50 (piso genérico). Ninguno necesita entrenar.
- **Salida:** `outputs/results/10_cluster_summary.json` + tabla (ARI, NMI, nº clusters hallados vs real, Rank-1, mAP).

> **Prerrequisito:** pushear al repo los archivos nuevos (`src/reid/encoders.py`, `src/reid/cluster.py`, `scripts/10_cluster_reid.py`, cambios en `config.py` y `reid_dataset.py`) en la rama que se clona abajo.

## 0. GPU

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'Sin GPU: Entorno de ejecución -> Cambiar tipo -> GPU.'
print('GPU:', torch.cuda.get_device_name(0))

## 1. Montar Drive

`DRIVE_ROOT` debe apuntar a la carpeta que contiene los `.zip`. Si la carpeta es **compartida**, agregá un acceso directo a *Mi unidad* (clic derecho → Organizar → Añadir acceso directo) y ajustá la ruta.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
# Ajustá esta ruta a la carpeta con los .zip (CMPD300_Baseline.zip, BeefCattle_Muzzle_Individualized.zip, ...).
DRIVE_ROOT = Path('/content/drive/MyDrive/cattle_reid')
assert DRIVE_ROOT.is_dir(), f'No existe {DRIVE_ROOT}. Ajustá DRIVE_ROOT.'
print('DRIVE_ROOT:', DRIVE_ROOT)
print('contenido:', [p.name for p in DRIVE_ROOT.iterdir()])

## 2. Traer el código (clon de tu repo/rama)

In [ ]:
import os, shutil
os.chdir('/content')
REPO_URL = 'https://github.com/benjaminvitale/tp-final-vision2-Pujol-Vitale.git'
BRANCH   = 'stage3-fase0'   # rama con los archivos de la Fase 0 (Etapa 3)
REPO_DIR = '/content/tp-final-vision2-Pujol-Vitale'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1

## 3. Dependencias

Colab ya trae torch/torchvision/scikit-learn. Solo falta `transformers` (para DINOv2). `sklearn.cluster.HDBSCAN` requiere scikit-learn ≥ 1.3 (Colab lo cumple).

In [ ]:
!pip -q install 'transformers==4.40.2'
import sklearn; print('sklearn', sklearn.__version__)
from sklearn.cluster import HDBSCAN  # falla si sklearn < 1.3
print('HDBSCAN disponible ✅')

## 4. Persistir outputs en Drive (symlink)

Así el `10_cluster_summary.json` y el cache de HuggingFace quedan en Drive.

In [ ]:
DRIVE_OUTPUTS = DRIVE_ROOT / 'outputs'
for sub in ('checkpoints', 'logs', 'results'):
    drive_sub = DRIVE_OUTPUTS / sub; drive_sub.mkdir(parents=True, exist_ok=True)
    local_sub = Path(REPO_DIR) / 'outputs' / sub
    if local_sub.exists() and not local_sub.is_symlink():
        shutil.rmtree(local_sub)
    if not local_sub.exists():
        os.symlink(drive_sub, local_sub, target_is_directory=True)
    print(f'outputs/{sub} -> {drive_sub}')

## 5. Extraer el Zenodo Muzzle DB (target)

Dataset chico (~643 MB); se extrae completo. Se autodetecta la carpeta con las 268 subcarpetas `cattle_*`.

In [ ]:
import zipfile
MUZZLE_ZIP = DRIVE_ROOT / 'BeefCattle_Muzzle_Individualized.zip'
assert MUZZLE_ZIP.is_file(), f'Falta {MUZZLE_ZIP}.'
MUZZLE_LOCAL = Path('/content/muzzle')
if not MUZZLE_LOCAL.exists():
    MUZZLE_LOCAL.mkdir(parents=True)
    !unzip -q "{MUZZLE_ZIP}" -d "{MUZZLE_LOCAL}"

def find_dataset_root(base: Path) -> Path:
    # carpeta que contiene subcarpetas de clase (cattle_*/ con imágenes)
    for p in [base, *[d for d in base.rglob('*') if d.is_dir()]]:
        subs = [d for d in p.iterdir() if d.is_dir()]
        if len(subs) >= 100 and any(d.name.lower().startswith('cattle') for d in subs):
            return p
    raise RuntimeError('No encuentro la carpeta con las clases cattle_*.')

TARGET_DIR = find_dataset_root(MUZZLE_LOCAL)
n_ids = len([d for d in TARGET_DIR.iterdir() if d.is_dir()])
print('TARGET_DIR:', TARGET_DIR, '| clases:', n_ids)
# muestra de nombres reales de archivo (para chequear el naming de sesión)
ej = next(d for d in TARGET_DIR.iterdir() if d.is_dir())
print('ejemplo', ej.name, '->', [f.name for f in list(ej.iterdir())[:8]])

## 5b. TAREA 0 — Gate: ¿existe estructura de sesión? (correr ANTES de clusterizar)

El Zenodo Muzzle DB nombra los archivos `cattle_XXXX_DSCF####.jpg` — `DSCF####` es el contador de la cámara. Esta celda decide si cada animal tiene **una** toma (DSCF contiguos → cross-sesión imposible, sobre-segmentación = problema de representación, control anti-fuga vía pHash) o **varias** (split disjunto por sesión válido). Sin GPU, tarda segundos.

In [ ]:
!python scripts/11_session_diag.py --target-dir "{TARGET_DIR}"

## 6. Correr la Fase 0 (DINOv2 + ImageNet)

Primero mirá el bloque **SESSION DIAGNOSTICS** en el log: dice si el split por sesión es real para este dataset (si `session_split_is_meaningful=False`, los nombres de archivo no codifican sesiones y el control anti-ráfaga es débil).

In [ ]:
!python scripts/10_cluster_reid.py \
    --target-dir "{TARGET_DIR}" \
    --encoders dinov2 imagenet \
    --batch-size 64

## 7. (Opcional) Sumar nuestros encoders de hocico

Si ya tenés `cmpd300_source.pt` (Fase 5) o el ArcFace en `outputs/checkpoints/` (vía el symlink a Drive), agregalos al mismo pipeline. Identidades disjuntas del source.

In [ ]:
CKPT = Path(REPO_DIR)/'outputs/checkpoints/cmpd300_source.pt'
if CKPT.is_file():
    !python scripts/10_cluster_reid.py \
        --target-dir "{TARGET_DIR}" \
        --encoders dinov2 imagenet resnet-ckpt \
        --ckpt "{CKPT}" --batch-size 64
else:
    print('No hay cmpd300_source.pt; salteando. (Corré la Fase 5 o copialo a Drive.)')

## 8. Resultados

In [ ]:
import json
p = Path('outputs/results/10_cluster_summary.json')
res = json.loads(p.read_text())
print('session_diagnostics:', json.dumps(res['session_diagnostics'], indent=2))
for e in res['encoders']:
    if 'error' in e:
        print(e['encoder'], 'ERROR', e['error']); continue
    cd = e['clustering_session_dedup']
    best = max(cd['dbscan_grid'], key=lambda r: r['ARI'])
    hdb = cd['hdbscan'] or {}
    r = e['retrieval_session_single_shot'] or {}
    print(f"{e['encoder']:20} DBSCAN ARI={best['ARI']} (eps={best['eps']}, "
          f"{best['n_clusters_found']}/{cd['n_clusters_true']} clust) "
          f"HDBSCAN ARI={hdb.get('ARI')} Rank-1={r.get('rank1')}")

## 9. Tests de perturbación — borde / rotación / ambas

Ablaciones controladas sobre las 268: se corrompe la entrada y se mide cuánto se mueve el embedding (**drift**) y cuánto cae la tarea (Rank-1, ARI).
- **borde**: conserva el hocico central, ruidea el anillo → ¿mira el hocico o el fondo?
- **rotación**: gira el hocico (sin esquinas negras) → ¿es invariante a la pose? (candidata a explicar la sobre-segmentación)
- **ambas**.

Mirá primero el PNG de ejemplos: si el crop es muy apretado y el 'borde' se come parte del hocico, subí `--keep` (p. ej. 0.7); si querés más/menos giro, cambiá `--angle`.

In [ ]:
!python scripts/12_perturbation_tests.py \
    --target-dir "{TARGET_DIR}" \
    --encoders dinov2 imagenet \
    --keep 0.55 --angle 20 --save-examples

from IPython.display import Image as IPImage
IPImage('outputs/results/12_perturb_examples.png')   # filas: clean, borde, rotación, ambas

Para sumar tu encoder de hocico al test (identidades disjuntas del source):

In [ ]:
CKPT = Path(REPO_DIR)/'outputs/checkpoints/cmpd300_source.pt'
if CKPT.is_file():
    !python scripts/12_perturbation_tests.py \
        --target-dir "{TARGET_DIR}" \
        --encoders dinov2 imagenet resnet-ckpt --ckpt "{CKPT}" \
        --keep 0.55 --angle 20
else:
    print('No hay cmpd300_source.pt; salteando.')

## Cómo leer esto

- **ARI/NMI** cerca de 1 = el encoder descubrió las identidades sin que le digan cuántas hay.
- **nº clusters hallados vs real (268):** ¿sobre- o sub-segmenta?
- **DBSCAN** = mejor del grid de eps (el grid completo está en el JSON; eps NO se elige mirando labels en producción). **HDBSCAN** = estimación automática. **k-means** conoce el k real → techo con trampa.
- **Tesis:** si DINOv2 (genérico) iguala o supera a nuestros encoders de hocico, la especialización no ayudó a descubrir identidades — y eso es un hallazgo, no un fracaso.